In [1]:
import numpy as np
import pandas as pd

import plotly.express as px

from sklearn.cluster import SpectralClustering
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# Clustering

In [2]:
# Config

INPUT_FILE = "../styles/fighter_career_stats.csv"
OUTPUT_FILE = "fighter_style_cluster.csv"

MIN_FIGHTS = 3
N_NEIGHBORS = 15
RANDOM_STATE = 100

K_VALUES = range(2, 16)


# Spectral clustering function

def run_spectral_clustering(
    input_file,
    K_CLUSTERS,
    MIN_FIGHTS,
    N_NEIGHBORS,
    RANDOM_STATE
):
    # 1. Load data

    df = pd.read_csv(input_file)

    df_model = df[df["total_fights"] >= MIN_FIGHTS].copy()

    # 2. Select clustering features

    exclude_cols = [
        "fighter",
        "total_fights",
        "total_fight_time"
    ]

    feature_cols = [
        col for col in df_model.columns
        if col not in exclude_cols
    ]

    feature_cols = [
        col for col in feature_cols
        if pd.api.types.is_numeric_dtype(df_model[col])
    ]

    # 3. Preprocess features

    X_raw = df_model[feature_cols].replace([np.inf, -np.inf], np.nan)

    preprocess = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    X_scaled = preprocess.fit_transform(X_raw)

    # SpectralClustering requires n_neighbors < n_samples
    adjusted_n_neighbors = min(N_NEIGHBORS, len(df_model) - 1)

    # 4. Spectral clustering

    spectral = SpectralClustering(
        n_clusters=K_CLUSTERS,
        affinity="nearest_neighbors",
        n_neighbors=adjusted_n_neighbors,
        assign_labels="kmeans",
        random_state=RANDOM_STATE
    )

    df_model[f"style_k_{K_CLUSTERS}"] = spectral.fit_predict(X_scaled)

    # 5. Cluster counts and summary

    cluster_counts = (
        df_model[f"style_k_{K_CLUSTERS}"]
        .value_counts()
        .sort_index()
    )

    cluster_summary = (
        df_model
        .groupby(f"style_k_{K_CLUSTERS}")[feature_cols]
        .mean()
        .round(3)
    )

    cluster_summary_table = cluster_summary.T
    cluster_summary_table.index.name = "feature"

    cluster_summary_table.columns = [
        f"Cluster {col}" for col in cluster_summary_table.columns
    ]

    fighter_style_output = df_model[[
        "fighter",
        f"style_k_{K_CLUSTERS}"
    ]].copy()

    return fighter_style_output, cluster_counts, cluster_summary_table


# Run spectral clustering for k = 2 through 10

fighter_style_cluster = None
cluster_results = {}

for k in K_VALUES:
    print(f"\nRunning spectral clustering for k = {k}")

    fighter_k_styles, cluster_counts, cluster_summary_table = run_spectral_clustering(
        input_file=INPUT_FILE,
        K_CLUSTERS=k,
        MIN_FIGHTS=MIN_FIGHTS,
        N_NEIGHBORS=N_NEIGHBORS,
        RANDOM_STATE=RANDOM_STATE
    )

    print(f"\nCluster counts for k = {k}:")
    print(cluster_counts)

    cluster_results[k] = {
        "cluster_counts": cluster_counts,
        "cluster_summary_table": cluster_summary_table
    }

    if fighter_style_cluster is None:
        fighter_style_cluster = fighter_k_styles
    else:
        fighter_style_cluster = fighter_style_cluster.merge(
            fighter_k_styles,
            on="fighter",
            how="inner"
        )


# Save final fighter style cluster file

fighter_style_cluster = fighter_style_cluster.sort_values("fighter").reset_index(drop=True)

print("\nFinal shape:", fighter_style_cluster.shape)

fighter_style_cluster.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved: {OUTPUT_FILE}")


Running spectral clustering for k = 2

Cluster counts for k = 2:
style_k_2
0    1089
1     766
Name: count, dtype: int64

Running spectral clustering for k = 3

Cluster counts for k = 3:
style_k_3
0     387
1     419
2    1049
Name: count, dtype: int64

Running spectral clustering for k = 4

Cluster counts for k = 4:
style_k_4
0    1030
1     231
2     346
3     248
Name: count, dtype: int64

Running spectral clustering for k = 5

Cluster counts for k = 5:
style_k_5
0    235
1    202
2    258
3    331
4    829
Name: count, dtype: int64

Running spectral clustering for k = 6

Cluster counts for k = 6:
style_k_6
0    257
1    782
2    386
3    133
4    117
5    180
Name: count, dtype: int64

Running spectral clustering for k = 7

Cluster counts for k = 7:
style_k_7
0    288
1    105
2    222
3    308
4    646
5    155
6    131
Name: count, dtype: int64

Running spectral clustering for k = 8

Cluster counts for k = 8:
style_k_8
0    543
1    317
2    248
3    215
4     77
5     86
6    1

# PreProcess before RF

In [3]:
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

FIGHT_FILE = "../styles/clean_total_fight_data.csv"
STYLE_FILE = "fighter_style_cluster.csv"

OUTPUT_FILE = "style_matchup_rf_data_all_k.csv"


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

fights = pd.read_csv(FIGHT_FILE)
fighter_styles = pd.read_csv(STYLE_FILE)

# Make sure names are clean strings
for col in ["winner", "R_fighter", "B_fighter"]:
    fights[col] = fights[col].astype(str).str.strip()

fighter_styles["fighter"] = fighter_styles["fighter"].astype(str).str.strip()


# ------------------------------------------------------------
# 2. Validate style columns
# ------------------------------------------------------------

style_cols = [f"style_k_{k}" for k in K_VALUES]

missing_style_cols = [
    col for col in style_cols
    if col not in fighter_styles.columns
]

if missing_style_cols:
    raise ValueError(f"Missing style columns in {STYLE_FILE}: {missing_style_cols}")


# ------------------------------------------------------------
# 3. Merge k=2...10 styles into fight-level data
# ------------------------------------------------------------

r_styles = fighter_styles[["fighter"] + style_cols].rename(columns={
    "fighter": "R_fighter",
    **{f"style_k_{k}": f"R_style_k_{k}" for k in K_VALUES}
})

b_styles = fighter_styles[["fighter"] + style_cols].rename(columns={
    "fighter": "B_fighter",
    **{f"style_k_{k}": f"B_style_k_{k}" for k in K_VALUES}
})

style_matchups = (
    fights[["winner", "R_fighter", "B_fighter"]]
    .merge(r_styles, on="R_fighter", how="left")
    .merge(b_styles, on="B_fighter", how="left")
)

print("\nRows before dropping missing styles:", len(style_matchups))

# Drop fights where either fighter is missing any style assignment
required_style_cols = (
    [f"R_style_k_{k}" for k in K_VALUES] +
    [f"B_style_k_{k}" for k in K_VALUES]
)

style_matchups = style_matchups.dropna(subset=required_style_cols).copy()

print("Rows after dropping missing styles:", len(style_matchups))


# ------------------------------------------------------------
# 4. Convert winner name into R/B target
# ------------------------------------------------------------

def get_winner_side(row):
    winner = str(row["winner"]).strip()
    r_fighter = str(row["R_fighter"]).strip()
    b_fighter = str(row["B_fighter"]).strip()

    if winner == r_fighter:
        return "R"
    elif winner == b_fighter:
        return "B"
    elif winner.lower() in ["r", "red", "r_fighter"]:
        return "R"
    elif winner.lower() in ["b", "blue", "b_fighter"]:
        return "B"
    else:
        return np.nan


style_matchups["winner_side"] = style_matchups.apply(get_winner_side, axis=1)

print("\nRows before dropping unknown winner:", len(style_matchups))

style_matchups = style_matchups.dropna(subset=["winner_side"]).copy()

print("Rows after dropping unknown winner:", len(style_matchups))


# ------------------------------------------------------------
# 5. Randomly assign neutral fighter_1 / fighter_2 sides
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

# True = keep R as fighter_1, B as fighter_2
# False = swap B into fighter_1, R into fighter_2
style_matchups["keep_original_order"] = rng.choice(
    [True, False],
    size=len(style_matchups),
    replace=True
)

style_matchups["fighter_1"] = np.where(
    style_matchups["keep_original_order"],
    style_matchups["R_fighter"],
    style_matchups["B_fighter"]
)

style_matchups["fighter_2"] = np.where(
    style_matchups["keep_original_order"],
    style_matchups["B_fighter"],
    style_matchups["R_fighter"]
)

# Target: did fighter_1 win?
style_matchups["fighter_1_won"] = np.where(
    (
        (style_matchups["keep_original_order"] & (style_matchups["winner_side"] == "R")) |
        (~style_matchups["keep_original_order"] & (style_matchups["winner_side"] == "B"))
    ),
    1,
    0
)


# ------------------------------------------------------------
# 6. Add fighter_1 / fighter_2 style columns for every k
# ------------------------------------------------------------

for k in K_VALUES:
    r_col = f"R_style_k_{k}"
    b_col = f"B_style_k_{k}"

    f1_col = f"fighter_1_style_k_{k}"
    f2_col = f"fighter_2_style_k_{k}"

    style_matchups[f1_col] = np.where(
        style_matchups["keep_original_order"],
        style_matchups[r_col],
        style_matchups[b_col]
    )

    style_matchups[f2_col] = np.where(
        style_matchups["keep_original_order"],
        style_matchups[b_col],
        style_matchups[r_col]
    )

    # Keep style IDs categorical
    style_matchups[f1_col] = style_matchups[f1_col].astype(int).astype(str)
    style_matchups[f2_col] = style_matchups[f2_col].astype(int).astype(str)


# ------------------------------------------------------------
# 7. Final RF dataset
# ------------------------------------------------------------

final_cols = [
    "fighter_1",
    "fighter_2",
    "fighter_1_won"
]

for k in K_VALUES:
    final_cols.extend([
        f"fighter_1_style_k_{k}",
        f"fighter_2_style_k_{k}"
    ])

rf_data_final = style_matchups[final_cols].copy()

# Shuffle final rows
rf_data_final = (
    rf_data_final
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)


print("\nTarget distribution:")
print(rf_data_final["fighter_1_won"].value_counts())



Rows before dropping missing styles: 8693
Rows after dropping missing styles: 7531

Rows before dropping unknown winner: 7531
Rows after dropping unknown winner: 7392

Target distribution:
fighter_1_won
1    3718
0    3674
Name: count, dtype: int64


# Random Forest

In [4]:
# ------------------------------------------------------------
# Random Forest comparison across k = 2...10
# ------------------------------------------------------------

# Random Forest function

def run_style_rf_for_k(
    rf_data_final,
    k,
    random_state=100,
    test_size=0.20,
    cv=5,
    print_results=True
):
    fighter_1_col = f"fighter_1_style_k_{k}"
    fighter_2_col = f"fighter_2_style_k_{k}"
    target_col = "fighter_1_won"

    required_cols = [fighter_1_col, fighter_2_col, target_col]

    missing_cols = [
        col for col in required_cols
        if col not in rf_data_final.columns
    ]

    if missing_cols:
        raise ValueError(f"Missing required columns for k={k}: {missing_cols}")

    # --------------------------------------------------------
    # Remove same-style matchups for this specific k
    # --------------------------------------------------------

    data_k = rf_data_final[required_cols].copy()

    data_k[fighter_1_col] = data_k[fighter_1_col].astype(str)
    data_k[fighter_2_col] = data_k[fighter_2_col].astype(str)

    rows_before = len(data_k)

    data_k = data_k[data_k[fighter_1_col] != data_k[fighter_2_col]].copy()

    rows_after = len(data_k)
    rows_removed = rows_before - rows_after

    if data_k[target_col].nunique() < 2:
        raise ValueError(f"Only one target class remains after filtering same-style matchups for k={k}.")

    # --------------------------------------------------------
    # Predictors and target
    # --------------------------------------------------------

    X = data_k[[fighter_1_col, fighter_2_col]].copy()
    y = data_k[target_col].copy()

    # --------------------------------------------------------
    # Train/test split
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    # --------------------------------------------------------
    # One-hot encode style IDs
    # --------------------------------------------------------

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocess_rf = ColumnTransformer(
        transformers=[
            ("style_ohe", encoder, [fighter_1_col, fighter_2_col])
        ],
        remainder="drop"
    )

    rf_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )

    rf_pipeline = Pipeline(steps=[
        ("preprocess", preprocess_rf),
        ("model", rf_model)
    ])

    # --------------------------------------------------------
    # Fit and evaluate train/test
    # --------------------------------------------------------

    rf_pipeline.fit(X_train, y_train)

    y_pred = rf_pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred, labels=[1, 0])

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------

    cv_scores = cross_val_score(
        rf_pipeline,
        X,
        y,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    result = {
        "k": k,
        "rows_before_filter": rows_before,
        "rows_after_filter": rows_after,
        "same_style_rows_removed": rows_removed,
        "target_1_rate": y.mean(),
        "test_accuracy": accuracy,
        "test_balanced_accuracy": balanced_accuracy,
        "cv_mean_balanced_accuracy": cv_scores.mean(),
        "cv_std_balanced_accuracy": cv_scores.std(),
        "cv_scores": cv_scores,
        "confusion_matrix": cm
    }

    if print_results:
        print("\n" + "-" * 60)
        print(f"k = {k}")
        print("-" * 60)

        print("Rows before same-style filter:", rows_before)
        print("Rows after same-style filter:", rows_after)
        print("Same-style rows removed:", rows_removed)

        print("\nTarget distribution:")
        print(y.value_counts(normalize=True).round(3))
        print(y.value_counts())

        print("\nTest accuracy:", round(accuracy, 3))
        print("Test balanced accuracy:", round(balanced_accuracy, 3))

        print("\nConfusion matrix:")
        print(cm)

        print(f"\n{CV}-fold CV balanced accuracy:")
        print(cv_scores.round(3))
        print("Mean:", round(cv_scores.mean(), 3))
        print("Std:", round(cv_scores.std(), 3))

    return result


# ------------------------------------------------------------
# Run RF for each k
# ------------------------------------------------------------

CV = 15

rf_results = []

for k in K_VALUES:
    result_k = run_style_rf_for_k(
        rf_data_final=rf_data_final,
        k=k,
        random_state=RANDOM_STATE,
        test_size=0.20,
        cv=CV,
        print_results=True
    )

    rf_results.append(result_k)


# ------------------------------------------------------------
# Compare all k values
# ------------------------------------------------------------

rf_results_table = pd.DataFrame([
    {
        "k": r["k"],
        "rows_before_filter": r["rows_before_filter"],
        "rows_after_filter": r["rows_after_filter"],
        "same_style_rows_removed": r["same_style_rows_removed"],
        "target_1_rate": round(r["target_1_rate"], 3),
        "test_accuracy": round(r["test_accuracy"], 3),
        "test_balanced_accuracy": round(r["test_balanced_accuracy"], 3),
        "cv_mean_balanced_accuracy": round(r["cv_mean_balanced_accuracy"], 3),
        "cv_std_balanced_accuracy": round(r["cv_std_balanced_accuracy"], 3)
    }
    for r in rf_results
])

rf_results_table = (
    rf_results_table
    .sort_values("cv_mean_balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("Random Forest performance by k")
print("=" * 80)

display(rf_results_table)

best_k = rf_results_table.loc[0, "k"]
best_cv_score = rf_results_table.loc[0, "cv_mean_balanced_accuracy"]

print(f"\nBest k by CV balanced accuracy: {best_k}")
print(f"Best CV balanced accuracy: {best_cv_score}")



------------------------------------------------------------
k = 2
------------------------------------------------------------
Rows before same-style filter: 7392
Rows after same-style filter: 2782
Same-style rows removed: 4610

Target distribution:
fighter_1_won
1    0.511
0    0.489
Name: proportion, dtype: float64
fighter_1_won
1    1421
0    1361
Name: count, dtype: int64

Test accuracy: 0.524
Test balanced accuracy: 0.525

Confusion matrix:
[[141 144]
 [121 151]]

15-fold CV balanced accuracy:
[0.564 0.532 0.469 0.41  0.458 0.537 0.538 0.518 0.535 0.534 0.507 0.552
 0.486 0.526 0.481]
Mean: 0.51
Std: 0.04

------------------------------------------------------------
k = 3
------------------------------------------------------------
Rows before same-style filter: 7392
Rows after same-style filter: 3832
Same-style rows removed: 3560

Target distribution:
fighter_1_won
1    0.503
0    0.497
Name: proportion, dtype: float64
fighter_1_won
1    1926
0    1906
Name: count, dtype: int64

,k,rows_before_filter,rows_after_filter,same_style_rows_removed,target_1_rate,test_accuracy,test_balanced_accuracy,cv_mean_balanced_accuracy,cv_std_balanced_accuracy
0,12,7392,6147,1245,0.500,0.613,0.613,0.616,0.027
1,10,7392,5992,1400,0.501,0.614,0.614,0.607,0.025
2,9,7392,5752,1640,0.502,0.615,0.615,0.604,0.020
3,8,7392,5610,1782,0.501,0.590,0.590,0.602,0.022
4,7,7392,5342,2050,0.503,0.606,0.606,0.602,0.023
5,5,7392,4604,2788,0.505,0.584,0.583,0.601,0.019
6,14,7392,6305,1087,0.504,0.590,0.590,0.598,0.026
7,4,7392,4010,3382,0.507,0.612,0.612,0.598,0.029
8,13,7392,6318,1074,0.500,0.619,0.619,0.597,0.021
9,11,7392,6057,1335,0.503,0.570,0.570,0.597,0.026



Best k by CV balanced accuracy: 12
Best CV balanced accuracy: 0.616


# Weighted Clustering

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import SpectralClustering


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

INPUT_FILE = "../styles/fighter_career_stats.csv"
OUTPUT_FILE = "weighted_fighter_style_cluster.csv"

MIN_FIGHTS = 3
N_NEIGHBORS = 15
RANDOM_STATE = 100

K_VALUES = range(2, 16)


# ------------------------------------------------------------
# Feature weighting
# ------------------------------------------------------------

HIGH_WEIGHT = 2.0
MID_WEIGHT = 1.0
LOW_WEIGHT = 0.5


def get_feature_weight(feature_name):
    """
    Assigns feature weights after scaling.
    """

    # Highest weight features
    if (
        feature_name == "fight_quickness"
        or feature_name.startswith("win_")
        or feature_name.endswith("_att_per_min")
        or feature_name in ["CTRL_pct", "KD_per_min", "SUB_ATT_per_min"]
    ):
        return HIGH_WEIGHT

    # Lowest weight features
    elif (
        feature_name.startswith("loss_")
        or feature_name.endswith("_absorbed_per_min")
        or feature_name.endswith("_opp_att_per_min")
    ):
        return LOW_WEIGHT

    # Middle weight features
    elif (
        feature_name.endswith("_acc")
        or feature_name == "REV_per_min"
        or feature_name.endswith("_land_per_min")
        or feature_name.endswith("_def")
    ):
        return MID_WEIGHT

    # Default to middle weight if a numeric feature is not explicitly categorized
    else:
        return MID_WEIGHT


# ------------------------------------------------------------
# Spectral clustering function
# ------------------------------------------------------------

def run_spectral_clustering(
    input_file,
    K_CLUSTERS,
    MIN_FIGHTS,
    N_NEIGHBORS,
    RANDOM_STATE
):
    # --------------------------------------------------------
    # 1. Load data
    # --------------------------------------------------------

    df = pd.read_csv(input_file)

    df_model = df[df["total_fights"] >= MIN_FIGHTS].copy()

    # --------------------------------------------------------
    # 2. Select clustering features
    # --------------------------------------------------------

    exclude_cols = [
        "fighter",
        "total_fights",
        "total_fight_time"
    ]

    feature_cols = [
        col for col in df_model.columns
        if col not in exclude_cols
    ]

    feature_cols = [
        col for col in feature_cols
        if pd.api.types.is_numeric_dtype(df_model[col])
    ]

    # --------------------------------------------------------
    # 3. Preprocess features
    # --------------------------------------------------------

    X_raw = df_model[feature_cols].replace([np.inf, -np.inf], np.nan)

    preprocess = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    X_scaled = preprocess.fit_transform(X_raw)

    # --------------------------------------------------------
    # 4. Apply feature weights after scaling
    # --------------------------------------------------------

    feature_weights = np.array([
        get_feature_weight(col)
        for col in feature_cols
    ])

    X_weighted = X_scaled * feature_weights

    weights_table = pd.DataFrame({
        "feature": feature_cols,
        "weight": feature_weights
    })

    # SpectralClustering requires n_neighbors < n_samples
    adjusted_n_neighbors = min(N_NEIGHBORS, len(df_model) - 1)

    # --------------------------------------------------------
    # 5. Spectral clustering
    # --------------------------------------------------------

    spectral = SpectralClustering(
        n_clusters=K_CLUSTERS,
        affinity="nearest_neighbors",
        n_neighbors=adjusted_n_neighbors,
        assign_labels="kmeans",
        random_state=RANDOM_STATE
    )

    df_model[f"style_k_{K_CLUSTERS}"] = spectral.fit_predict(X_weighted)

    # --------------------------------------------------------
    # 6. Cluster counts and summary
    # --------------------------------------------------------

    cluster_counts = (
        df_model[f"style_k_{K_CLUSTERS}"]
        .value_counts()
        .sort_index()
    )

    cluster_summary = (
        df_model
        .groupby(f"style_k_{K_CLUSTERS}")[feature_cols]
        .mean()
        .round(3)
    )

    cluster_summary_table = cluster_summary.T
    cluster_summary_table.index.name = "feature"

    cluster_summary_table.columns = [
        f"Cluster {col}" for col in cluster_summary_table.columns
    ]

    fighter_style_output = df_model[[
        "fighter",
        f"style_k_{K_CLUSTERS}"
    ]].copy()

    return fighter_style_output, cluster_counts, cluster_summary_table, weights_table


# ------------------------------------------------------------
# Run spectral clustering for k = 2 through 15
# ------------------------------------------------------------

fighter_style_cluster = None
cluster_results = {}
weights_table = None

for k in K_VALUES:
    print(f"\nRunning weighted spectral clustering for k = {k}")

    fighter_k_styles, cluster_counts, cluster_summary_table, weights_table = run_spectral_clustering(
        input_file=INPUT_FILE,
        K_CLUSTERS=k,
        MIN_FIGHTS=MIN_FIGHTS,
        N_NEIGHBORS=N_NEIGHBORS,
        RANDOM_STATE=RANDOM_STATE
    )

    print(f"\nCluster counts for k = {k}:")
    print(cluster_counts)

    cluster_results[k] = {
        "cluster_counts": cluster_counts,
        "cluster_summary_table": cluster_summary_table
    }

    if fighter_style_cluster is None:
        fighter_style_cluster = fighter_k_styles
    else:
        fighter_style_cluster = fighter_style_cluster.merge(
            fighter_k_styles,
            on="fighter",
            how="inner"
        )


# ------------------------------------------------------------
# Save final fighter style cluster file
# ------------------------------------------------------------

fighter_style_cluster = (
    fighter_style_cluster
    .sort_values("fighter")
    .reset_index(drop=True)
)

print("\nFinal shape:", fighter_style_cluster.shape)

fighter_style_cluster.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved: {OUTPUT_FILE}")




Running weighted spectral clustering for k = 2

Cluster counts for k = 2:
style_k_2
0    1101
1     754
Name: count, dtype: int64

Running weighted spectral clustering for k = 3

Cluster counts for k = 3:
style_k_3
0     398
1     407
2    1050
Name: count, dtype: int64

Running weighted spectral clustering for k = 4

Cluster counts for k = 4:
style_k_4
0    258
1    394
2    939
3    264
Name: count, dtype: int64

Running weighted spectral clustering for k = 5

Cluster counts for k = 5:
style_k_5
0    808
1    255
2    348
3    202
4    242
Name: count, dtype: int64

Running weighted spectral clustering for k = 6

Cluster counts for k = 6:
style_k_6
0     95
1    426
2    671
3    148
4    128
5    387
Name: count, dtype: int64

Running weighted spectral clustering for k = 7

Cluster counts for k = 7:
style_k_7
0    107
1    211
2    248
3    136
4    674
5    121
6    358
Name: count, dtype: int64

Running weighted spectral clustering for k = 8

Cluster counts for k = 8:
style_k_8
0

# RF Weighted PreProcess

In [8]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

FIGHT_FILE = "../styles/clean_total_fight_data.csv"
STYLE_FILE = "weighted_fighter_style_cluster.csv"

OUTPUT_FILE = "weighted_style_matchup_rf_data_all_k.csv"


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

fights = pd.read_csv(FIGHT_FILE)
fighter_styles = pd.read_csv(STYLE_FILE)

# Make sure names are clean strings
for col in ["winner", "R_fighter", "B_fighter"]:
    fights[col] = fights[col].astype(str).str.strip()

fighter_styles["fighter"] = fighter_styles["fighter"].astype(str).str.strip()


# ------------------------------------------------------------
# 2. Validate style columns
# ------------------------------------------------------------

style_cols = [f"style_k_{k}" for k in K_VALUES]

missing_style_cols = [
    col for col in style_cols
    if col not in fighter_styles.columns
]

if missing_style_cols:
    raise ValueError(f"Missing style columns in {STYLE_FILE}: {missing_style_cols}")


# ------------------------------------------------------------
# 3. Merge k=2...10 styles into fight-level data
# ------------------------------------------------------------

r_styles = fighter_styles[["fighter"] + style_cols].rename(columns={
    "fighter": "R_fighter",
    **{f"style_k_{k}": f"R_style_k_{k}" for k in K_VALUES}
})

b_styles = fighter_styles[["fighter"] + style_cols].rename(columns={
    "fighter": "B_fighter",
    **{f"style_k_{k}": f"B_style_k_{k}" for k in K_VALUES}
})

style_matchups = (
    fights[["winner", "R_fighter", "B_fighter"]]
    .merge(r_styles, on="R_fighter", how="left")
    .merge(b_styles, on="B_fighter", how="left")
)

print("\nRows before dropping missing styles:", len(style_matchups))

# Drop fights where either fighter is missing any style assignment
required_style_cols = (
    [f"R_style_k_{k}" for k in K_VALUES] +
    [f"B_style_k_{k}" for k in K_VALUES]
)

style_matchups = style_matchups.dropna(subset=required_style_cols).copy()

print("Rows after dropping missing styles:", len(style_matchups))


# ------------------------------------------------------------
# 4. Convert winner name into R/B target
# ------------------------------------------------------------

def get_winner_side(row):
    winner = str(row["winner"]).strip()
    r_fighter = str(row["R_fighter"]).strip()
    b_fighter = str(row["B_fighter"]).strip()

    if winner == r_fighter:
        return "R"
    elif winner == b_fighter:
        return "B"
    elif winner.lower() in ["r", "red", "r_fighter"]:
        return "R"
    elif winner.lower() in ["b", "blue", "b_fighter"]:
        return "B"
    else:
        return np.nan


style_matchups["winner_side"] = style_matchups.apply(get_winner_side, axis=1)

print("\nRows before dropping unknown winner:", len(style_matchups))

style_matchups = style_matchups.dropna(subset=["winner_side"]).copy()

print("Rows after dropping unknown winner:", len(style_matchups))


# ------------------------------------------------------------
# 5. Randomly assign neutral fighter_1 / fighter_2 sides
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

# True = keep R as fighter_1, B as fighter_2
# False = swap B into fighter_1, R into fighter_2
style_matchups["keep_original_order"] = rng.choice(
    [True, False],
    size=len(style_matchups),
    replace=True
)

style_matchups["fighter_1"] = np.where(
    style_matchups["keep_original_order"],
    style_matchups["R_fighter"],
    style_matchups["B_fighter"]
)

style_matchups["fighter_2"] = np.where(
    style_matchups["keep_original_order"],
    style_matchups["B_fighter"],
    style_matchups["R_fighter"]
)

# Target: did fighter_1 win?
style_matchups["fighter_1_won"] = np.where(
    (
        (style_matchups["keep_original_order"] & (style_matchups["winner_side"] == "R")) |
        (~style_matchups["keep_original_order"] & (style_matchups["winner_side"] == "B"))
    ),
    1,
    0
)


# ------------------------------------------------------------
# 6. Add fighter_1 / fighter_2 style columns for every k
# ------------------------------------------------------------

for k in K_VALUES:
    r_col = f"R_style_k_{k}"
    b_col = f"B_style_k_{k}"

    f1_col = f"fighter_1_style_k_{k}"
    f2_col = f"fighter_2_style_k_{k}"

    style_matchups[f1_col] = np.where(
        style_matchups["keep_original_order"],
        style_matchups[r_col],
        style_matchups[b_col]
    )

    style_matchups[f2_col] = np.where(
        style_matchups["keep_original_order"],
        style_matchups[b_col],
        style_matchups[r_col]
    )

    # Keep style IDs categorical
    style_matchups[f1_col] = style_matchups[f1_col].astype(int).astype(str)
    style_matchups[f2_col] = style_matchups[f2_col].astype(int).astype(str)


# ------------------------------------------------------------
# 7. Final RF dataset
# ------------------------------------------------------------

final_cols = [
    "fighter_1",
    "fighter_2",
    "fighter_1_won"
]

for k in K_VALUES:
    final_cols.extend([
        f"fighter_1_style_k_{k}",
        f"fighter_2_style_k_{k}"
    ])

rf_data_final = style_matchups[final_cols].copy()

# Shuffle final rows
rf_data_final = (
    rf_data_final
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)


print("\nTarget distribution:")
print(rf_data_final["fighter_1_won"].value_counts())



Rows before dropping missing styles: 8693
Rows after dropping missing styles: 7531

Rows before dropping unknown winner: 7531
Rows after dropping unknown winner: 7392

Target distribution:
fighter_1_won
1    3718
0    3674
Name: count, dtype: int64


# RF weighted

In [9]:
# ------------------------------------------------------------
# Run RF for each k
# ------------------------------------------------------------

CV = 15

rf_results = []

for k in K_VALUES:
    result_k = run_style_rf_for_k(
        rf_data_final=rf_data_final,
        k=k,
        random_state=RANDOM_STATE,
        test_size=0.20,
        cv=CV,
        print_results=True
    )

    rf_results.append(result_k)


# ------------------------------------------------------------
# Compare all k values
# ------------------------------------------------------------

rf_results_table = pd.DataFrame([
    {
        "k": r["k"],
        "rows_before_filter": r["rows_before_filter"],
        "rows_after_filter": r["rows_after_filter"],
        "same_style_rows_removed": r["same_style_rows_removed"],
        "target_1_rate": round(r["target_1_rate"], 3),
        "test_accuracy": round(r["test_accuracy"], 3),
        "test_balanced_accuracy": round(r["test_balanced_accuracy"], 3),
        "cv_mean_balanced_accuracy": round(r["cv_mean_balanced_accuracy"], 3),
        "cv_std_balanced_accuracy": round(r["cv_std_balanced_accuracy"], 3)
    }
    for r in rf_results
])

rf_results_table = (
    rf_results_table
    .sort_values("cv_mean_balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("Random Forest performance by k")
print("=" * 80)

display(rf_results_table)

best_k = rf_results_table.loc[0, "k"]
best_cv_score = rf_results_table.loc[0, "cv_mean_balanced_accuracy"]

print(f"\nBest k by CV balanced accuracy: {best_k}")
print(f"Best CV balanced accuracy: {best_cv_score}")



------------------------------------------------------------
k = 2
------------------------------------------------------------
Rows before same-style filter: 7392
Rows after same-style filter: 2779
Same-style rows removed: 4613

Target distribution:
fighter_1_won
1    0.512
0    0.488
Name: proportion, dtype: float64
fighter_1_won
1    1422
0    1357
Name: count, dtype: int64

Test accuracy: 0.541
Test balanced accuracy: 0.541

Confusion matrix:
[[153 132]
 [123 148]]

15-fold CV balanced accuracy:
[0.544 0.521 0.491 0.409 0.482 0.539 0.535 0.513 0.557 0.517 0.518 0.553
 0.464 0.531 0.482]
Mean: 0.51
Std: 0.038

------------------------------------------------------------
k = 3
------------------------------------------------------------
Rows before same-style filter: 7392
Rows after same-style filter: 3566
Same-style rows removed: 3826

Target distribution:
fighter_1_won
1    0.5
0    0.5
Name: proportion, dtype: float64
fighter_1_won
1    1784
0    1782
Name: count, dtype: int64

T

,k,rows_before_filter,rows_after_filter,same_style_rows_removed,target_1_rate,test_accuracy,test_balanced_accuracy,cv_mean_balanced_accuracy,cv_std_balanced_accuracy
0,13,7392,6327,1065,0.502,0.604,0.604,0.591,0.024
1,5,7392,4644,2748,0.499,0.595,0.595,0.590,0.036
2,6,7392,4859,2533,0.503,0.579,0.579,0.588,0.034
3,7,7392,5132,2260,0.502,0.581,0.581,0.584,0.032
4,12,7392,6280,1112,0.507,0.572,0.572,0.583,0.028
5,14,7392,6401,991,0.501,0.562,0.562,0.581,0.022
6,4,7392,4227,3165,0.499,0.593,0.593,0.577,0.026
7,15,7392,6391,1001,0.501,0.573,0.573,0.576,0.019
8,8,7392,5740,1652,0.505,0.585,0.585,0.573,0.025
9,10,7392,6063,1329,0.508,0.583,0.583,0.571,0.025



Best k by CV balanced accuracy: 13
Best CV balanced accuracy: 0.591
